# CSE301 Machine Learning Capstone — Clustering Track
## Heart Failure / Cardiovascular Risk Dataset

**Algorithms:** K-Means and Agglomerative Hierarchical Clustering  
**Dataset:** `HeartFailureDataset.csv`  
**Random seed:** 42

### Important methodology
- `id` is an identifier, so it is excluded from clustering.
- `cardio` is the ground-truth outcome. It is **not used to fit either clustering algorithm**; it is used only after clustering for optional interpretation.
- The data has 70,000 rows. Agglomerative clustering and dendrogram construction can be expensive, so both algorithms are compared on the **same reproducible sample** (default 5,000 rows). Change `SAMPLE_SIZE` only if your computer has enough memory/time.
- Outliers are treated by IQR clipping for continuous features. Binary/category-coded features are not clipped.
- Clustering is unsupervised: do not interpret cluster IDs as disease diagnoses.

In [ ]:
# If needed, install packages in a separate cell:
# %pip install pandas numpy matplotlib seaborn scikit-learn scipy

import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans, AgglomerativeClustering
from sklearn.decomposition import PCA
from sklearn.manifold import TSNE
from sklearn.metrics import (
    silhouette_score,
    davies_bouldin_score,
    calinski_harabasz_score
)
from scipy.cluster.hierarchy import linkage, dendrogram

RANDOM_STATE = 42
SAMPLE_SIZE = 5000
sns.set_theme(style="whitegrid")

## 1. Load and audit the dataset

Keep the CSV in the same folder as this notebook, or update `DATA_PATH`.

In [ ]:
DATA_PATH = "HeartFailureDataset.csv"
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)
display(df.head())
print("\nData types:")
display(df.dtypes.to_frame("dtype"))
print("\nMissing values per column:")
display(df.isna().sum().to_frame("missing_count"))
print("\nDuplicate rows:", df.duplicated().sum())
print("\nTarget distribution (cardio; for interpretation only):")
display(df["cardio"].value_counts(dropna=False).rename_axis("cardio").to_frame("count"))

## 2. Initial data checks and feature engineering

In this dataset, `age` is recorded in days. Convert it to years for easier interpretation and create BMI from height and weight. Remove the identifier and target from the clustering feature matrix.

In [ ]:
data = df.copy()
data = data.drop_duplicates().reset_index(drop=True)

# Convert age from days to years
if "age" in data.columns:
    data["age_years"] = data["age"] / 365.25
    data = data.drop(columns=["age"])

# Height is in cm and weight in kg
if {"height", "weight"}.issubset(data.columns):
    height_m = data["height"] / 100
    data["bmi"] = data["weight"] / (height_m ** 2)
    data = data.drop(columns=["height", "weight"])

# Exclude identifier and ground-truth target from unsupervised model fitting
target = data["cardio"].copy() if "cardio" in data.columns else None
drop_cols = [c for c in ["id", "cardio"] if c in data.columns]
X_raw = data.drop(columns=drop_cols).copy()

print("Feature columns used for clustering:")
print(X_raw.columns.tolist())
print("\nFeature matrix shape:", X_raw.shape)
display(X_raw.describe().T)

## 3. Exploratory data analysis (EDA)

### Feature distributions

In [ ]:
X_raw.hist(figsize=(16, 12), bins=30, edgecolor="black")
plt.suptitle("Feature Distributions Before Outlier Treatment", y=1.02, fontsize=16)
plt.tight_layout()
plt.show()

### Correlation heatmap

Correlation is shown for numeric features. Strong correlation does not necessarily imply that a feature should be removed; use domain reasoning.

In [ ]:
plt.figure(figsize=(12, 8))
sns.heatmap(X_raw.corr(numeric_only=True), cmap="coolwarm", center=0, annot=True, fmt=".2f")
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.show()

## 4. Missing-value handling

Report missing values first, then impute numeric features with the median. This is robust to skewed distributions. The current file may have no missing values; the imputer keeps the workflow reusable.

In [ ]:
print("Missing values before imputation:")
display(X_raw.isna().sum().to_frame("missing_count"))

imputer = SimpleImputer(strategy="median")
X_imputed = pd.DataFrame(
    imputer.fit_transform(X_raw),
    columns=X_raw.columns,
    index=X_raw.index
)

print("Missing values after imputation:", int(X_imputed.isna().sum().sum()))

## 5. Outlier detection with boxplots — before treatment

IQR limits are calculated for continuous measurements. Binary and category-coded variables (`gender`, `cholesterol`, `gluc`, `smoke`, `alco`, `active`) are excluded from clipping because their values represent categories rather than continuous measurements.

In [ ]:
categorical_or_binary = [
    c for c in ["gender", "cholesterol", "gluc", "smoke", "alco", "active"]
    if c in X_imputed.columns
]
continuous_cols = [c for c in X_imputed.columns if c not in categorical_or_binary]

def plot_boxplots(frame, columns, title, max_cols=4):
    if not columns:
        print("No continuous columns available for boxplots.")
        return
    ncols = min(max_cols, len(columns))
    nrows = int(np.ceil(len(columns) / ncols))
    fig, axes = plt.subplots(nrows, ncols, figsize=(4*ncols, 3.5*nrows))
    axes = np.array(axes).reshape(-1)
    for ax, col in zip(axes, columns):
        sns.boxplot(y=frame[col], ax=ax, color="skyblue")
        ax.set_title(col)
    for ax in axes[len(columns):]:
        ax.set_visible(False)
    fig.suptitle(title, fontsize=15)
    plt.tight_layout()
    plt.show()

plot_boxplots(X_imputed, continuous_cols, "Continuous Feature Boxplots — Before Outlier Treatment")

### Count outliers using the 1.5 × IQR rule

In [ ]:
def iqr_outlier_summary(frame, columns):
    rows = []
    for col in columns:
        q1 = frame[col].quantile(0.25)
        q3 = frame[col].quantile(0.75)
        iqr = q3 - q1
        lower = q1 - 1.5 * iqr
        upper = q3 + 1.5 * iqr
        count = int(((frame[col] < lower) | (frame[col] > upper)).sum())
        rows.append({"feature": col, "lower_bound": lower, "upper_bound": upper,
                     "outlier_count": count})
    return pd.DataFrame(rows)

outlier_before = iqr_outlier_summary(X_imputed, continuous_cols)
display(outlier_before)

## 6. Outlier treatment and boxplots — after treatment

Clip continuous features to their IQR bounds instead of deleting rows. This preserves the same observations for both clustering algorithms while limiting the influence of extreme values. Review the before/after plots and explain this choice in the presentation.

In [ ]:
X_clean = X_imputed.copy()
iqr_bounds = {}

for col in continuous_cols:
    q1 = X_clean[col].quantile(0.25)
    q3 = X_clean[col].quantile(0.75)
    iqr = q3 - q1
    lower = q1 - 1.5 * iqr
    upper = q3 + 1.5 * iqr
    iqr_bounds[col] = (lower, upper)
    X_clean[col] = X_clean[col].clip(lower=lower, upper=upper)

plot_boxplots(X_clean, continuous_cols, "Continuous Feature Boxplots — After IQR Clipping")
outlier_after = iqr_outlier_summary(X_clean, continuous_cols)
display(outlier_after)

## 7. Scale features and create a reproducible modelling sample

Scaling is important because the algorithms use distances. StandardScaler gives features a mean of approximately zero and standard deviation of approximately one. The sample is drawn once and reused by both algorithms for a fair comparison.

In [ ]:
scaler = StandardScaler()
X_scaled_all = pd.DataFrame(
    scaler.fit_transform(X_clean),
    columns=X_clean.columns,
    index=X_clean.index
)

sample_n = min(SAMPLE_SIZE, len(X_scaled_all))
sample_idx = X_scaled_all.sample(n=sample_n, random_state=RANDOM_STATE).index
X_cluster = X_scaled_all.loc[sample_idx].reset_index(drop=True)

# Ground-truth labels are retained only for post-clustering interpretation
y_reference = target.loc[sample_idx].reset_index(drop=True) if target is not None else None

print("Rows used by BOTH clustering algorithms:", X_cluster.shape[0])
print("Number of clustering features:", X_cluster.shape[1])
display(X_cluster.head())

## 8. K-Means — choose the number of clusters

Use the elbow curve (inertia vs. k) and silhouette score to help select k. The elbow is a judgement rather than an automatic guarantee.

In [ ]:
k_values = range(2, 8)
inertias = []
silhouettes = []

for k in k_values:
    km = KMeans(n_clusters=k, random_state=RANDOM_STATE, n_init=10)
    labels_k = km.fit_predict(X_cluster)
    inertias.append(km.inertia_)
    silhouettes.append(silhouette_score(X_cluster, labels_k))

fig, ax1 = plt.subplots(figsize=(8, 5))
ax1.plot(list(k_values), inertias, marker="o", label="Inertia")
ax1.set_xlabel("Number of clusters (k)")
ax1.set_ylabel("Inertia")
ax1.set_title("K-Means Elbow Curve")
ax1.legend()
plt.tight_layout()
plt.show()

display(pd.DataFrame({"k": list(k_values), "inertia": inertias,
                      "silhouette_score": silhouettes}))

### Fit the final K-Means model

Set `K_FINAL` after inspecting the elbow curve and scores. The default is 3; change it if your evidence supports a different value.

In [ ]:
K_FINAL = 3

kmeans = KMeans(n_clusters=K_FINAL, random_state=RANDOM_STATE, n_init=10)
kmeans_labels = kmeans.fit_predict(X_cluster)
print("K-Means cluster sizes:")
display(pd.Series(kmeans_labels, name="cluster").value_counts().sort_index().to_frame("count"))

## 9. Agglomerative Hierarchical Clustering

Compare common linkage strategies on the same scaled sample. We evaluate each candidate using silhouette score and use the best-scoring linkage for the chosen k. The dendrogram is computed on a smaller subset for readability and to reduce memory use.

In [ ]:
linkage_results = []
for link in ["ward", "complete", "average"]:
    # Ward linkage requires Euclidean distance; sklearn's default metric is Euclidean.
    model = AgglomerativeClustering(n_clusters=K_FINAL, linkage=link)
    labels_link = model.fit_predict(X_cluster)
    linkage_results.append({
        "linkage": link,
        "silhouette_score": silhouette_score(X_cluster, labels_link),
        "davies_bouldin_index": davies_bouldin_score(X_cluster, labels_link),
        "calinski_harabasz_index": calinski_harabasz_score(X_cluster, labels_link)
    })

linkage_table = pd.DataFrame(linkage_results).sort_values("silhouette_score", ascending=False)
display(linkage_table)

BEST_LINKAGE = linkage_table.iloc[0]["linkage"]
print("Selected linkage based on silhouette score:", BEST_LINKAGE)

agglo = AgglomerativeClustering(n_clusters=K_FINAL, linkage=BEST_LINKAGE)
agglo_labels = agglo.fit_predict(X_cluster)

print("Agglomerative cluster sizes:")
display(pd.Series(agglo_labels, name="cluster").value_counts().sort_index().to_frame("count"))

### Dendrogram

Hierarchical linkage is calculated on a smaller sample because dendrogram computation becomes expensive as the number of rows increases. This dendrogram illustrates the merging structure; the final Agglomerative model above uses the shared modelling sample.

In [ ]:
DENDROGRAM_SIZE = min(500, len(X_cluster))
X_dendro = X_cluster.sample(n=DENDROGRAM_SIZE, random_state=RANDOM_STATE)
Z = linkage(X_dendro, method=BEST_LINKAGE, metric="euclidean")

plt.figure(figsize=(14, 6))
dendrogram(Z, truncate_mode="lastp", p=30, leaf_rotation=45, leaf_font_size=9)
plt.title(f"Agglomerative Clustering Dendrogram ({BEST_LINKAGE} linkage; display truncated)")
plt.xlabel("Merged samples / cluster size")
plt.ylabel("Euclidean distance")
plt.tight_layout()
plt.show()

## 10. Clustering evaluation and comparison table

All three required internal metrics are calculated on the same rows:
- **Silhouette Score:** higher is better (range −1 to 1).
- **Davies–Bouldin Index:** lower is better.
- **Calinski–Harabasz Index:** higher is generally better.

These are internal clustering metrics, not classification accuracy.

In [ ]:
comparison = pd.DataFrame([
    {
        "Algorithm": "K-Means",
        "Parameters": f"k={K_FINAL}",
        "Silhouette Score": silhouette_score(X_cluster, kmeans_labels),
        "Davies-Bouldin Index": davies_bouldin_score(X_cluster, kmeans_labels),
        "Calinski-Harabasz Index": calinski_harabasz_score(X_cluster, kmeans_labels)
    },
    {
        "Algorithm": "Agglomerative Hierarchical",
        "Parameters": f"k={K_FINAL}, linkage={BEST_LINKAGE}",
        "Silhouette Score": silhouette_score(X_cluster, agglo_labels),
        "Davies-Bouldin Index": davies_bouldin_score(X_cluster, agglo_labels),
        "Calinski-Harabasz Index": calinski_harabasz_score(X_cluster, agglo_labels)
    }
])

display(comparison.round(4))

## 11. Mandatory dimensionality reduction: PCA visualisation

PCA reduces the scaled feature space to two principal components for visualisation. PCA is fitted to the modelling sample; the cluster labels are overlaid afterward.

In [ ]:
pca = PCA(n_components=2, random_state=RANDOM_STATE)
X_pca = pca.fit_transform(X_cluster)
print("Explained variance ratio:", pca.explained_variance_ratio_)
print("Total variance explained by PC1 + PC2:", round(pca.explained_variance_ratio_.sum(), 4))

pca_df = pd.DataFrame({
    "PC1": X_pca[:, 0],
    "PC2": X_pca[:, 1],
    "K-Means Cluster": kmeans_labels.astype(str),
    "Agglomerative Cluster": agglo_labels.astype(str)
})

fig, axes = plt.subplots(1, 2, figsize=(14, 5))
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="K-Means Cluster",
                palette="tab10", s=18, alpha=0.7, ax=axes[0], linewidth=0)
axes[0].set_title("PCA — K-Means Clusters")
sns.scatterplot(data=pca_df, x="PC1", y="PC2", hue="Agglomerative Cluster",
                palette="tab10", s=18, alpha=0.7, ax=axes[1], linewidth=0)
axes[1].set_title("PCA — Agglomerative Clusters")
plt.tight_layout()
plt.show()

## 12. Optional t-SNE visualisation

t-SNE is encouraged by the rubric. It is more computationally expensive and is intended for visualisation, not for evaluating the clustering. Run on a smaller subset for speed.

In [ ]:
TSNE_SIZE = min(1500, len(X_cluster))
tsne_idx = np.random.RandomState(RANDOM_STATE).choice(len(X_cluster), size=TSNE_SIZE, replace=False)
X_tsne_input = X_cluster.iloc[tsne_idx]
labels_tsne = kmeans_labels[tsne_idx]

tsne = TSNE(n_components=2, random_state=RANDOM_STATE, init="pca",
            learning_rate="auto", perplexity=min(30, max(5, (TSNE_SIZE - 1) // 3)))
X_tsne = tsne.fit_transform(X_tsne_input)

tsne_df = pd.DataFrame({
    "t-SNE 1": X_tsne[:, 0],
    "t-SNE 2": X_tsne[:, 1],
    "K-Means Cluster": labels_tsne.astype(str)
})
plt.figure(figsize=(8, 6))
sns.scatterplot(data=tsne_df, x="t-SNE 1", y="t-SNE 2",
                hue="K-Means Cluster", palette="tab10", s=25, alpha=0.75, linewidth=0)
plt.title("t-SNE Visualisation of K-Means Clusters (subset)")
plt.tight_layout()
plt.show()

## 13. Optional ground-truth interpretation (not used in fitting)

The `cardio` target was excluded from all clustering features and model fitting. This crosstab is only a post-hoc check of how clusters relate to the existing labels; it is not a clustering evaluation metric and should not be described as proof that clusters predict disease.

In [ ]:
if y_reference is not None:
    interpretation = pd.DataFrame({
        "K-Means Cluster": kmeans_labels,
        "Agglomerative Cluster": agglo_labels,
        "cardio_reference": y_reference
    })
    print("K-Means cluster vs. reference cardio label:")
    display(pd.crosstab(interpretation["K-Means Cluster"],
                        interpretation["cardio_reference"], normalize="index").round(3))
    print("Agglomerative cluster vs. reference cardio label:")
    display(pd.crosstab(interpretation["Agglomerative Cluster"],
                        interpretation["cardio_reference"], normalize="index").round(3))

## 14. Final summary and conclusion prompts

Use the actual values from the comparison table when writing your conclusion. Do not claim one algorithm is best without referring to the metrics and plots.

In your own words, answer:
1. Which algorithm has the higher silhouette score?
2. Which has the lower Davies–Bouldin Index?
3. Which has the higher Calinski–Harabasz Index?
4. Does PCA show clearly separated groups, or is there overlap?
5. How did IQR clipping and standardisation affect the workflow?
6. Why was `cardio` excluded during clustering?
7. Why was a reproducible sample used for comparing both algorithms?

**Reproducibility note:** If your instructor requires the full 70,000 rows for K-Means, you can fit K-Means on `X_scaled_all`, but retain a documented shared sample for a fair computational comparison with Agglomerative clustering.